# Bir defalık lokal hazırlık: splits + Dataset801

Bu notebook büyük Dataset801 çıktısını **lokal SSD'ye** yazar. Google Drive'a yalnızca küçük `splits.json` yedeği gönderilir. Böylece hazırlama işlemi Drive yazma hızını beklemez.

In [19]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO = Path('/Users/anil/Desktop/GitHub/ToothFairy3-IAC-Segmentation-Flow')
GDRIVE = Path("/Users/anil/Library/CloudStorage/GoogleDrive-anil04keskin@gmail.com/Drive'ım")
DRIVE_RUNS = GDRIVE / 'ToothFairy/ToothFairy3/iac_runs'
TF3_SOURCE = GDRIVE / 'ToothFairy/ToothFairy3/ToothFairy3'
LOCAL_DATASET = Path.home() / 'tf_work/Dataset801_IAC_LR'

assert (REPO / 'data/create_folds.py').is_file(), REPO
assert (TF3_SOURCE / 'imagesTr').is_dir(), TF3_SOURCE
assert (TF3_SOURCE / 'labelsTr').is_dir(), TF3_SOURCE
assert (TF3_SOURCE / 'dataset.json').is_file(), TF3_SOURCE

free_gb = shutil.disk_usage(Path.home()).free / 1024**3
print(f'Lokal boş alan: {free_gb:.1f} GB')
assert free_gb >= 45, 'En az 45 GB boş lokal alan öneriliyor'
LOCAL_DATASET.parent.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
print('Kaynak       :', TF3_SOURCE)
print('Lokal çıktı  :', LOCAL_DATASET)
print('Drive runs   :', DRIVE_RUNS)


Lokal boş alan: 52.3 GB
Kaynak       : /Users/anil/Library/CloudStorage/GoogleDrive-anil04keskin@gmail.com/Drive'ım/ToothFairy/ToothFairy3/ToothFairy3
Lokal çıktı  : /Users/anil/tf_work/Dataset801_IAC_LR
Drive runs   : /Users/anil/Library/CloudStorage/GoogleDrive-anil04keskin@gmail.com/Drive'ım/ToothFairy/ToothFairy3/iac_runs


## 1. Split üret

Aynı vaka listesi, kod, 5 fold ve seed `1234` ile deterministiktir. TrackA farklı bir split ile eğitildiyse burada üretilen dosyayı kullanmadan önce TrackA'nın `splits_final.json` dosyasıyla karşılaştır.

In [20]:
splits = REPO / 'configs/splits.json'
subprocess.run([
    sys.executable, 'data/create_folds.py',
    '--src', str(TF3_SOURCE),
    '--out', str(splits),
    '--seed', '1234',
], check=True)

drive_split = DRIVE_RUNS / 'configs_cache/splits.json'
drive_split.parent.mkdir(parents=True, exist_ok=True)
drive_split.write_bytes(splits.read_bytes())
print('Lokal split:', splits)
print('Drive yedeği:', drive_split)


[ids] P=417 F=63 S=52
  fold 0: train=383 val=97 (F in val=13)
  fold 1: train=383 val=97 (F in val=13)
  fold 2: train=384 val=96 (F in val=13)
  fold 3: train=385 val=95 (F in val=12)
  fold 4: train=385 val=95 (F in val=12)
[splits] written -> /Users/anil/Desktop/GitHub/ToothFairy3-IAC-Segmentation-Flow/configs/splits.json  (external OOD test S=52)
Lokal split: /Users/anil/Desktop/GitHub/ToothFairy3-IAC-Segmentation-Flow/configs/splits.json
Drive yedeği: /Users/anil/Library/CloudStorage/GoogleDrive-anil04keskin@gmail.com/Drive'ım/ToothFairy/ToothFairy3/iac_runs/configs_cache/splits.json


## 2. Dataset801'i lokal SSD'ye hazırla

Önceki Drive hedefli hücrenin hâlâ çalışmadığından emin ol. Bu hücre tamamlandığında `[done] kept 480; dropped 0` beklenir.

In [21]:
subprocess.run([
    sys.executable, 'data/prepare_iac_dataset.py',
    '--src', str(TF3_SOURCE),
    '--dst', str(LOCAL_DATASET),
    '--subset', 'PF',
    '--copy-images',
    '--workers', '5',
], check=True)


[ids] resolved by name -> Left=3  Right=4
[cases] 480 paired image/label cases (subset=PF)
[done] kept 480; dropped 0
[dataset.json] written: /Users/anil/tf_work/Dataset801_IAC_LR/dataset.json


CompletedProcess(args=['/usr/local/bin/python3', 'data/prepare_iac_dataset.py', '--src', "/Users/anil/Library/CloudStorage/GoogleDrive-anil04keskin@gmail.com/Drive'ım/ToothFairy/ToothFairy3/ToothFairy3", '--dst', '/Users/anil/tf_work/Dataset801_IAC_LR', '--subset', 'PF', '--copy-images', '--workers', '5'], returncode=0)

## 3. Sonucu doğrula

In [22]:
import json
split_data = json.loads(splits.read_text())
expected = len(split_data['development'])
n_images = len(list((LOCAL_DATASET / 'imagesTr').glob('*_0000.nii.gz')))
n_labels = len(list((LOCAL_DATASET / 'labelsTr').glob('*.nii.gz')))
assert (LOCAL_DATASET / 'dataset.json').is_file(), 'dataset.json eksik'
assert n_images == expected, f'imagesTr: {n_images}/{expected}'
assert n_labels == expected, f'labelsTr: {n_labels}/{expected}'
assert splits.read_bytes() == drive_split.read_bytes(), 'Lokal ve Drive split farklı'
print(f'HAZIR: {n_images} image + {n_labels} label')
print('Sonraki notebook: notebooks/mac_local_oof_sdf.ipynb')


HAZIR: 480 image + 480 label
Sonraki notebook: notebooks/mac_local_oof_sdf.ipynb


## 4. Wi-Fi varken Dataset801'i Drive'a kalıcı aktar

Bu hücre yalnızca eksik dosyaları kopyalar; mevcut yarım Drive klasörünü silmez. Mobil veriyle çalıştırma.

In [26]:
DRIVE_DATASET = DRIVE_RUNS / 'dataset_cache/Dataset801_IAC_LR'
DRIVE_DATASET.mkdir(parents=True, exist_ok=True)
t0 = __import__('time').monotonic()
# Google Drive Desktop sahiplik/izin metadata'sini desteklemez; -a kullanma.
# Kaynak dosyaları, Drive uygulaması senkronu tamamlanmadan asla silme.
subprocess.run([
    'rsync', '-rt', '--ignore-existing',
    f'{LOCAL_DATASET}/', f'{DRIVE_DATASET}/'
], check=True)
drive_images = len(list((DRIVE_DATASET / 'imagesTr').glob('*_0000.nii.gz')))
drive_labels = len(list((DRIVE_DATASET / 'labelsTr').glob('*.nii.gz')))
assert drive_images == expected and drive_labels == expected, (drive_images, drive_labels, expected)
print(f'Drive Dataset801 hazır: {drive_images} image + {drive_labels} label | {( __import__("time").monotonic()-t0)/60:.1f} min')


KeyboardInterrupt: 